In [2]:
import sys
sys.path.append("../..")
from src.sawmill.sawmill import Sawmill
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)

~/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
s = Sawmill("../../datasets_raw/tpc-ds/work_mem_2_256kB_2_128kB.log", workdir="../datasets/tpc-ds")
s.parse(regex_dict={"ID" : r'(?<=EDT \[ )\S{14}',"Timestamp": r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d{3}'}, force=True, message_prefix=r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}.\d{3}')

Initialized Sawmill with log file ../../datasets_raw/tpc-ds/work_mem_2_256kB_2_128kB.log
Work directory set to ../datasets/tpc-ds
Parsing file: ../../datasets_raw/tpc-ds/work_mem_2_256kB_2_128kB.log


Extracting variables from each line...: 100%|██████████| 98/98 [00:00<00:00, 452.29it/s]


Variables generated from regexes: 2
Variables generated by Drain: 158
Templates with at least 1 non-regex variable: 98
Templates with at least 2 occurrences: 98


Casting datetime variables...:   0%|          | 0/1 [00:00<?, ?it/s]~/.local/lib/python3.11/site-packages/tqdm/std.py:915: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return func(*args, **kwargs)
Dumping _parsed_variables to pkl file...: 100%|██████████| 1/1 [00:00<00:00, 1151.33it/s]

Parsing complete in 0.849597 seconds!


'0.849597'

In [4]:
s._parsed_variables.head(15)

,Name,Tag,Type,Occurrences,Preceding 3 tokens,Examples,From regex
0,ID,ID,str,2320,[],"[65427493.1d08f, 6542778d.1d421, 65427a85.1d8e9, 65428941.1dc31]",True
1,Timestamp,Timestamp,date,2320,[],"[2023-11-01 11:53:55.462, 2023-11-01 11:53:55.473, 2023-11-01 11:53:55.475, 2023-11-01 11:53:55.486, 2023-11-01 11:53:55.487]",True
2,3dad7fac_22,port,num,4,"[127.0.0.1, port, =]","[36560, 49382, 46396, 54196]",False
3,ce7e95f9_4,ce7e95f9_4,str,4,"[EDT, [, <*0>]","[3/600, 4/873, 3/1565, 4/2241]",False
4,ecde4441_4,ecde4441_4,str,4,"[EDT, [, <*0>]","[3/600, 4/873, 3/1565, 4/2241]",False
5,be64f0a7_4,be64f0a7_4,str,768,"[EDT, [, <*0>]","[3/601, 3/602, 3/603, 3/604, 3/605]",False
6,be64f0a7_11,statement,str,768,"[:, statement, :]","[BEGIN, COMMIT]",False
7,caf9d557_4,caf9d557_4,str,1152,"[EDT, [, <*0>]","[3/601, 3/0, 3/602, 3/603, 3/604]",False
8,caf9d557_11,duration,num,1152,"[:, duration, :]","[0.086, 0.063, 0.030, 0.043, 1036.105]",False
9,55a934b8_4,55a934b8_4,str,4,"[EDT, [, <*0>]","[3/601, 4/874, 3/1566, 4/2242]",False


In [4]:
s.set_causal_unit("ID")
s.prepare(count_occurences=True, force = True)

Causal unit set to ID (tag: ID)
Calculating aggregates for each causal unit...


Dumping _prepared_variables to pkl file...: 100%|██████████| 1/1 [00:00<00:00, 444.59it/s]

Successfully prepared the log with causal unit ID 
            (tag: ID)
Preparation complete in 1.291458 seconds!


'1.291458'

In [6]:
c = s.explore_candidate_causes("caf9d557_11+mean")
c.show()

,Candidate,ATE,P-value,Standard Error,Tag
0,55a934b8_15+mean,-81.600202,0.000001,[0.08819106042459803],55a934b8_15
1,Timestamp+earliest,0.000005,0.166943,[2.173718329941621e-06],Timestamp
2,ecde4441_4+latest=3/1565,6977.303084,0.421482,[6956.175861384832],ecde4441_4
3,be64f0a7_4+latest=3/1661,6977.303084,0.421482,[6956.175861384832],be64f0a7_4
4,55a934b8_4+latest=3/1566,6977.303084,0.421482,[6956.175861384832],55a934b8_4


In [7]:
c.inspect(0)

Information about prepared variable 55a934b8_15+mean:

--> Variable Information about 55a934b8_15:


,Name,Tag,Type,Occurrences,Preceding 3 tokens,Examples,From regex,Aggregates
10,55a934b8_15,55a934b8_15,num,4,"[work_mem, =, ']","[256, 128]",False,"[mean, min, max]"


--> Template Information about 55a934b8:


,TemplateText,TemplateId,Occurrences,VariableIndices,RegexIndices
5,<*1> EDT [ <*0> <*> ] postgres@tpcds1 LOG : statement : SET work_mem = ' <*> ' ;,55a934b8,4,"[4, 15]","[3, 0]"


--> Causal Unit Partial Information:


,cui,caf9d557_11+mean (outcome),55a934b8_15+mean (candidate)
0,65427493.1d08f,2646.117191,256.0
1,6542778d.1d421,2638.161483,256.0
2,65427a85.1d8e9,13097.529594,128.0
3,65428941.1dc31,13076.400854,128.0


In [8]:
c.accept(0)

''